# Module 3 Homework – Data Warehouse (BigQuery)

Dataset: NYC Yellow Taxi Trip Records (Jan–Jun 2024).

This notebook contains the SQL queries used and the final answers for the homework questions.


## Setup (External + Materialized)

Adjust the identifiers below to match your environment (`<PROJECT>`, `<DATASET>`, `<BUCKET>`).

```sql
-- External table (Parquet files in GCS)
CREATE OR REPLACE EXTERNAL TABLE `<PROJECT>.<DATASET>.yellow_taxi_external`
OPTIONS (
  format = 'PARQUET',
  uris = ['gs://<BUCKET>/yellow_tripdata_2024-*.parquet']
);

-- Materialized/native table (no partitioning/clustering)
CREATE OR REPLACE TABLE `<PROJECT>.<DATASET>.yellow_taxi_native` AS
SELECT *
FROM `<PROJECT>.<DATASET>.yellow_taxi_external`;
```


## Question 1
**What is count of records for the 2024 Yellow Taxi Data?**

```sql
SELECT COUNT(*) AS records
FROM `<PROJECT>.<DATASET>.yellow_taxi_native`;
```

**Answer:** `20,332,093`


## Question 2
**Estimated bytes read (External vs Materialized) for counting distinct PULocationID**

```sql
-- Native/materialized table
SELECT COUNT(DISTINCT PULocationID) AS distinct_pu
FROM `<PROJECT>.<DATASET>.yellow_taxi_native`;

-- External table
SELECT COUNT(DISTINCT PULocationID) AS distinct_pu
FROM `<PROJECT>.<DATASET>.yellow_taxi_external`;
```

**Answer:** `0 MB` (External Table) and `155.12 MB` (Materialized Table)


## Question 3
**Why are the estimated number of Bytes different?**

```sql
SELECT PULocationID
FROM `<PROJECT>.<DATASET>.yellow_taxi_native`;

SELECT PULocationID, DOLocationID
FROM `<PROJECT>.<DATASET>.yellow_taxi_native`;
```

**Answer:** BigQuery is columnar and only scans the columns requested; selecting 2 columns reads more data than selecting 1.


## Question 4
**How many records have a fare_amount of 0?**

```sql
SELECT COUNT(*) AS zero_fare_trips
FROM `<PROJECT>.<DATASET>.yellow_taxi_native`
WHERE fare_amount = 0;
```

**Answer:** `8,333`


## Question 5
**Best strategy if queries always filter by `tpep_dropoff_datetime` and order by `VendorID`**

```sql
CREATE OR REPLACE TABLE `<PROJECT>.<DATASET>.yellow_taxi_optimized`
PARTITION BY DATE(tpep_dropoff_datetime)
CLUSTER BY VendorID AS
SELECT *
FROM `<PROJECT>.<DATASET>.yellow_taxi_native`;
```

**Answer:** Partition by `tpep_dropoff_datetime` and Cluster on `VendorID`


## Question 6
**Distinct VendorIDs between `2024-03-01` and `2024-03-15` (inclusive) and estimated bytes**

```sql
-- Non-partitioned (native/materialized)
SELECT DISTINCT VendorID
FROM `<PROJECT>.<DATASET>.yellow_taxi_native`
WHERE tpep_dropoff_datetime >= '2024-03-01'
  AND tpep_dropoff_datetime <= '2024-03-15';

-- Partitioned + clustered
SELECT DISTINCT VendorID
FROM `<PROJECT>.<DATASET>.yellow_taxi_optimized`
WHERE tpep_dropoff_datetime >= '2024-03-01'
  AND tpep_dropoff_datetime <= '2024-03-15';
```

**Answer:** `310.24 MB` (non-partitioned) and `26.84 MB` (partitioned)


## Question 7
**Where is the data stored in the External Table you created?**

**Answer:** GCP Bucket (Google Cloud Storage)


## Question 8
**It is best practice in BigQuery to always cluster your data**

**Answer:** False


## Question 9 (not graded)

```sql
SELECT COUNT(*)
FROM `<PROJECT>.<DATASET>.yellow_taxi_native`;
```

**Estimated bytes:** `0 B`.

**Why:** BigQuery can answer `COUNT(*)` using table metadata (without scanning the columns).
